# The effect of periodic boundary conditions in MD

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

A finite simulation box can only hold a handful of particles, but we usually want
to model a piece of a much larger (effectively infinite) system. **Periodic
boundary conditions (PBC)** achieve this: the box is tiled infinitely in every
direction, a particle that leaves one side re-enters the opposite side, and — the
crucial part — each particle interacts with the **nearest image** of every other
particle, which may be the copy *across a boundary* rather than the one inside the
box. This nearest-copy rule is the **minimum-image convention**.

`LJ-ELEC_MD-Verlet.py` (the standard MD script) applies the minimum-image
convention in both its energy and force routines, via

```python
tmp = tmp - SignR(halfbox, tmp - halfbox) - SignR(halfbox, tmp + halfbox)
```

`LJ-ELEC_MD-Verlet-noPBC.py` is the same MD **with that one line removed** from the
energy and force loops (distances use the raw in-box coordinate difference). This
notebook shows what that change does.

**What we will find (and it is subtle):**

- The minimum-image convention only matters for pairs that interact *across* a
  boundary. We demonstrate this cleanly with **(A)** a static pair potential and
  **(B)** a controlled two-particle collision that happens *through* a wall.
- For the standard 20-particle system, LJ + Coulomb bind the particles into a
  droplet in the centre of the box that **never reaches the walls**, so PBC on vs
  off are **indistinguishable** here — demo **(C)**. Turning PBC off only bites
  when your system actually touches the boundary (a dense fluid, a gas filling the
  box, a small box).

Only stdlib `math`/`random` plus **matplotlib** are needed.

## 1. Imports

In [ ]:
# Colab-friendly install guard (only matplotlib is 3rd-party)
import importlib.util, subprocess, sys
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

import math
import random
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.animation import FuncAnimation

rc('animation', html='jshtml', embed_limit=64)   # MB; keep all animation frames
%matplotlib inline

## 2. Energy, force and the minimum-image switch

These are the exact LJ + Coulomb energy/force expressions from the MD script. The
**only** change is that the minimum-image correction is wrapped in a single helper,
`minimum_image`, gated by a `use_pbc` flag — so we can run *identical* code with the
convention on or off. With `use_pbc=False` the pair displacement is just the raw
coordinate difference (exactly what `LJ-ELEC_MD-Verlet-noPBC.py` does).

In [ ]:
def minimum_image(d, L, use_pbc):
    '''Nearest-image displacement along one axis. With use_pbc=False, return d as-is.'''
    if not use_pbc:
        return d
    if d >  0.5 * L:
        d -= L
    elif d < -0.5 * L:
        d += L
    return d

def pair_disp(ci, cj, box, use_pbc):
    '''Displacement cj - ci in x and y, with optional minimum image.'''
    return [minimum_image(cj[k] - ci[k], box[k], use_pbc) for k in range(2)]

# --- energy (from squared distance), same forms as the MD script ---
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1.0 / distsquare) ** 3 * rmin_exp6
    return epsilon * Z * (Z - 1)

def Coulomb2(distsquare, dielec, qa, qb):
    return qa * qb / (dielec * math.sqrt(distsquare))

def Calc_Ene(coord, epsilon, rmin, dielec, cutoffsquare, box, use_pbc, elec=1):
    Ene = ELJ = ECoul = 0.0
    rmin6 = rmin ** 6
    for i in range(len(coord) - 1):
        for j in range(i + 1, len(coord)):
            dx, dy = pair_disp(coord[i], coord[j], box, use_pbc)
            d2 = dx * dx + dy * dy
            if d2 < cutoffsquare:
                d2 = max(d2, 1e-6)
                vdw = LJ2(d2, epsilon, rmin6); Ene += vdw; ELJ += vdw
                if elec:
                    cc = Coulomb2(d2, dielec, coord[i][2], coord[j][2]); Ene += cc; ECoul += cc
    return Ene, ELJ, ECoul

# --- forces (from squared distance), same forms as the MD script ---
def ForceLJ2(d2, epsilon, rmin6, xi):
    rij = math.sqrt(d2); Z = (1.0 / d2) ** 3 * rmin6
    return epsilon * (2 * Z - 1) * (rmin6 * (-6.0 / rij ** 7)) * (xi / rij)

def ForceCoulomb(d2, dielec, qa, qb, xi):
    rij = math.sqrt(d2)
    return -1.0 * (qa * qb / dielec) * (1.0 / d2) * (xi / rij)

def Calc_Force(coord, epsilon, rmin, dielec, cutoffsquare, box, use_pbc):
    Force = []; rmin6 = rmin ** 6
    for i in range(len(coord)):
        fx = fy = 0.0
        for j in range(len(coord)):
            if i == j:
                continue
            dispx, dispy = pair_disp(coord[i], coord[j], box, use_pbc)
            d2 = dispx * dispx + dispy * dispy
            if d2 < cutoffsquare:
                d2 = max(d2, 1e-6)
                # displacement components used by the force (raw, matching the MD script)
                tx, ty = coord[j][0] - coord[i][0], coord[j][1] - coord[i][1]
                qa, qb = coord[i][2], coord[j][2]
                fx += ForceLJ2(d2, epsilon, rmin6, tx) + ForceCoulomb(d2, dielec, qa, qb, tx)
                fy += ForceLJ2(d2, epsilon, rmin6, ty) + ForceCoulomb(d2, dielec, qa, qb, ty)
        Force.append([fx, fy])
    return Force

# --- Verlet integrator + kinetic energy/temperature (from the MD script) ---
def Step1(coord, vel, force, h, mass):
    return [[coord[i][0] + h*vel[i][0] + 0.5*h*h*force[i][0]/mass,
             coord[i][1] + h*vel[i][1] + 0.5*h*h*force[i][1]/mass,
             coord[i][2]] for i in range(len(coord))]

def Verlet(coord, force, h, old, mass):
    return [[2*coord[i][0] - old[i][0] + force[i][0]/mass*h*h,
             2*coord[i][1] - old[i][1] + force[i][1]/mass*h*h,
             coord[i][2]] for i in range(len(coord))]

def CalcVel(old, coord, h):
    return [[(coord[i][0]-old[i][0])/(2*h), (coord[i][1]-old[i][1])/(2*h)]
            for i in range(len(coord))]

def calc_temp(vel, nat, k, mass):
    v2 = sum(vx*vx + vy*vy for vx, vy in vel)   # (this is the sum the noPBC .py bug got wrong)
    kin = 0.5 * mass * v2
    return kin, kin / (nat * k)

def charge_color(charge):
    return "#FFFFFF" if charge > 0 else "#333333" 

## 3. Parameters

The molecular-system and MD parameters used in this notebook, in the same format as
the other MD/EM notebooks. The demos all share one Lennard-Jones + Coulomb system:
**Demo C** runs the full 20-particle set below, while the controlled **Demos A–B**
fix one or two particles (and Demo B uses a large charge) to make the boundary effect
obvious. Values are in the toy model's arbitrary units.

### The molecular system and its properties

| Parameter | Meaning | Value used |
|---|---|---|
| `nAtoms` | number of particles (Demo C; Demos A–B place 1–2 fixed particles) | 20 |
| `Radius` | particle radius (drawn size, and default charge scale) | 25 |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` / `box` | periodic box dimensions | `[500, 500]` |
| `Epsilon` / `eps` | LJ well depth (van der Waals strength) | 25 |
| `Dielec` / `dielec` | dielectric constant (electrostatic screening) | 1 (vacuum) |
| `qat` | absolute charge per atom | `Radius` (Demo B uses 200 to force a strong deflection) |
| `frac_neg` | fraction of negative charges (Demo C) | 0.5 |
| `CutOff` | non-bonded cutoff — here equal to **half the box**, so the minimum-image convention bites exactly at the boundary | 250 |
| `Seed` | random seed for the initial configuration / velocities (Demo C) | 100 |

### Molecular dynamics

| Parameter | Meaning | Value used |
|---|---|---|
| `Mass` | particle mass (enters $a = F/m$) | 10 |
| `dt` (`timestep`) | MD integration time step $h$ | 2e-3 |
| `Temperature` | temperature of the Maxwell initial velocities (Demo C) | 300 K |
| `nsteps` / `nrun` | number of MD steps (Demo B / Demo C) | 7000 / 2000 |
| `use_pbc` | **the switch this notebook is about** — apply the minimum-image convention (`True`) or use raw in-box distances (`False`) | both, compared |

## 4. Demo A — the minimum-image pair potential

Put one particle **fixed near the right wall** ($x=475$) and slide a partner of the
**opposite** charge across the whole box at constant height. Plot the pair energy it
feels as a function of its $x$ position, once with the minimum-image convention
(**PBC**) and once with raw coordinates (**no PBC**).

- **With PBC** the potential is **periodic and continuous**: even when the sliding
  particle is at the *left* wall, its nearest image of the partner is just across the
  right boundary, so it still feels a well. The particle is never "alone".
- **Without PBC** the partner is only seen when the *in-box* distance is below the
  cutoff. Near the left wall the raw distance is ~450 (> cutoff), so the interaction
  simply **switches off** — a spurious, discontinuous loss of interaction at the
  boundary.

In [ ]:
box = [500.0, 500.0]; eps = 25.0; Radius = 25.0; Rmin = 2.24 * Radius
rmin6 = Rmin ** 6; dielec = 1.0; cutoffsq = 250.0 ** 2
fixed = [475.0, 306.0, +25.0]     # partner near the RIGHT wall
yslide = 250.0                    # sliding particle height (dy ~ Rmin at closest)
qslide = -25.0                    # opposite charge -> attractive well

xs = [i * 0.5 for i in range(1001)]   # 0 .. 500
def pair_E(x, use_pbc):
    test = [x, yslide, qslide]
    dx, dy = pair_disp(fixed, test, box, use_pbc)
    d2 = dx*dx + dy*dy
    if d2 >= cutoffsq:
        return None                    # beyond cutoff: no interaction
    d2 = max(d2, 1e-6)
    return LJ2(d2, eps, rmin6) + Coulomb2(d2, dielec, fixed[2], qslide)

fig, ax = plt.subplots(figsize=(9, 4.5))
for use_pbc, col, lab in ((True, "#1f77b4", "with PBC (minimum image)"),
                          (False, "#d62728", "no PBC (raw distance)")):
    E = [pair_E(x, use_pbc) for x in xs]
    ax.plot(xs, E, color=col, lw=2, label=lab)
ax.axvline(475, color="k", ls=":", lw=1); ax.text(478, ax.get_ylim()[1]*0.9, "partner", fontsize=8)
ax.axvline(0, color="grey", ls="--", lw=0.8); ax.axvline(500, color="grey", ls="--", lw=0.8)
ax.set_xlabel("x position of sliding particle"); ax.set_ylabel("pair energy")
ax.set_title("Pair energy across the box — PBC wraps the interaction; no-PBC drops it at the wall")
ax.legend(); plt.tight_layout(); plt.show()

near_left = 25.0
print(f"Sliding particle near the LEFT wall (x={near_left}):")
print(f"  with PBC : E = {pair_E(near_left, True):.2f}  (feels the partner across the boundary)")
print(f"  no  PBC  : E = {pair_E(near_left, False)}     (partner out of range -> no interaction)")

## 5. Demo B — a collision *through* the boundary

Now make it dynamical. Two particles of the **same** charge start near opposite
walls and move toward each other **through** the periodic boundary (one exits right
and wraps to the left, meeting the other). We integrate the real Verlet dynamics,
once with PBC and once without, from the identical start.

- **With PBC** the two particles *see each other across the wall*: they repel,
  exchange kinetic and potential energy, and **visibly veer apart** — a genuine
  collision that happens across the boundary.
- **Without PBC** they are invisible to each other (their raw separation stays above
  the cutoff), so they **cruise past undeflected** with a perfectly **flat energy** —
  unphysical.

In [ ]:
Mass = 10.0; dt = 2.0e-3; nsteps = 7000
cstboltz = 1000 * 0.00198722 / 4.18

def run_pair(use_pbc):
    # two LIKE charges near opposite walls, offset by dy=60, approaching slowly through
    # the boundary. The charge is large (q=200) so the Coulomb repulsion deflects them
    # strongly -- but at a separation still > Rmin, so no hard LJ collision.
    coord = [[455.0, 220.0, 200.0], [45.0, 280.0, 200.0]]
    vel   = [[3.0, 0.0], [-3.0, 0.0]]
    old = None; Et = []; traj = []
    for it in range(nsteps):
        Ene, _, _ = Calc_Ene(coord, eps, Rmin, dielec, cutoffsq, box, use_pbc)
        F = Calc_Force(coord, eps, Rmin, dielec, cutoffsq, box, use_pbc)
        if it == 0:
            old = [list(c) for c in coord]; coord = Step1(coord, vel, F, dt, Mass)
        else:
            tmp = coord; coord = Verlet(coord, F, dt, old, Mass)
            vel = CalcVel(old, coord, dt); old = tmp
        Kin, _ = calc_temp(vel, 2, cstboltz, Mass)
        Et.append(Ene + Kin); traj.append([list(c[:2]) for c in coord])
        # wrap positions at the box edges (done in BOTH versions, exactly like the scripts)
        for pp in range(len(coord)):
            for i in range(2):
                if coord[pp][i] < 0:        coord[pp][i] += box[i]; old[pp][i] += box[i]
                if coord[pp][i] > box[i]:   coord[pp][i] -= box[i]; old[pp][i] -= box[i]
    return Et, traj

Et_pbc,  tr_pbc  = run_pair(True)
Et_nopbc, tr_nopbc = run_pair(False)

t = [i * dt for i in range(nsteps)]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(t, Et_pbc,  color="#1f77b4", label="with PBC")
ax[0].plot(t, Et_nopbc, color="#d62728", label="no PBC")
ax[0].set_xlabel("time"); ax[0].set_ylabel("total energy"); ax[0].legend()
ax[0].set_title("Total energy")
ax[1].plot(t, [f[0][1] for f in tr_pbc],  color="#1f77b4", lw=1.3, label="p0 (PBC)")
ax[1].plot(t, [f[1][1] for f in tr_pbc],  color="#1f77b4", lw=1.3, ls="--", label="p1 (PBC)")
ax[1].plot(t, [f[0][1] for f in tr_nopbc], color="#d62728", lw=1.3, label="p0 (no PBC)")
ax[1].plot(t, [f[1][1] for f in tr_nopbc], color="#d62728", lw=1.3, ls="--", label="p1 (no PBC)")
ax[1].set_xlabel("time"); ax[1].set_ylabel("y position"); ax[1].legend(fontsize=8)
ax[1].set_title("y positions — PBC pair veers apart, no-PBC stays put")
plt.tight_layout(); plt.show()
print(f"Energy swing  with PBC: {max(Et_pbc)-min(Et_pbc):8.1f}   (they interact across the wall)")
print(f"Energy swing  no  PBC: {max(Et_nopbc)-min(Et_nopbc):8.1f}   (flat: they never see each other)")
dY = lambda tr: (tr[-1][0][1]-tr[0][0][1])
print(f"transverse deflection of p0  with PBC: {dY(tr_pbc):6.1f}   no PBC: {dY(tr_nopbc):6.1f}")

Animation of the same two runs side by side. Watch the PBC pair (left) repel
and **veer apart** as they meet through the wall — one exits the right edge and
re-enters on the left — while the no-PBC pair (right) glides past one another
untouched.

In [ ]:
stride = max(1, nsteps // 120)
frames = list(range(0, nsteps, stride))
fig, ax = plt.subplots(1, 2, figsize=(10, 5.2))
pts = []
for a, title, tr in ((ax[0], "with PBC", tr_pbc), (ax[1], "no PBC", tr_nopbc)):
    a.set_xlim(0, box[0]); a.set_ylim(0, box[1]); a.set_aspect("equal")
    a.set_facecolor("#ccddff"); a.set_xticks([]); a.set_yticks([]); a.set_title(title)
    s = a.scatter([p[0] for p in tr[0]], [p[1] for p in tr[0]],
                  s=400, c=["#333333", "#333333"], edgecolors="k")
    pts.append(s)
sup = fig.suptitle("", fontsize=12); plt.tight_layout()

def update(n):
    pts[0].set_offsets(tr_pbc[n]); pts[1].set_offsets(tr_nopbc[n])
    sup.set_text(f"t = {n*dt:.2f}")
    return pts
anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
plt.close(fig); anim

## 6. Demo C — the full 20-particle MD: here PBC makes no difference

Finally, the actual system the MD scripts simulate: 20 LJ + Coulomb particles with
the **shared parameters** (`nAtoms=20, Epsilon=25, qat=Radius, Seed=100`). We run it
both ways from the identical start and overlay the energies.

They lie on top of each other. The reason is dynamical, not because PBC is unused:
the minimum-image convention *is* active here (it adds ~37 distant cross-boundary
pairs), but those pairs sit near the cutoff where LJ + Coulomb are weak and, with
mixed charges, largely **cancel** — shifting the total energy by only ~0.1 %. More
importantly, at these parameters the particles settle into a **caged arrangement and
merely vibrate in place** (net displacement of only a few tens of units in a
500-wide box), and **no
particle ever crosses a box wall**. So the one thing that would make no-PBC blow up —
the force discontinuity when a particle **wraps** across a boundary — simply never
happens. **PBC (and its absence) only bite when particles actually cross, or strongly
interact across, a boundary** — a hotter/longer run, a denser system, or a smaller
box.

> This is also the resolution of a red herring: the original
> `LJ-ELEC_MD-Verlet-noPBC.py` *looked* dramatically different (≈25 % energy drift),
> but that was a **bug** in its `calc_temp` — the kinetic-energy sum was accidentally
> de-indented so only the last atom counted — not the missing PBC. With that fixed
> (and the parameters aligned to the PBC script) the two runs agree, as below.

In [ ]:
nAtoms = 20; Epsilon = 25.0; qat = Radius; frac_neg = 0.5
Temperature = 300.0; Seed = 100; nrun = 2000

def InitConf(n, dim, radius, qat, frac_neg, seed):
    random.seed(seed)
    coord = []; nneg = int(n * frac_neg)
    def place(charge):
        while True:
            x = random.random() * (dim[0] - 2*radius) + radius
            y = random.random() * (dim[1] - 2*radius) + radius
            if all((x-c[0])**2 + (y-c[1])**2 >= (2*radius)**2 for c in coord):
                coord.append([x, y, charge]); return
    for _ in range(nneg):        place(-qat)
    for _ in range(n - nneg):    place(+qat)
    return coord

def InitVel(n, temperature, cstboltz, mass, seed=1):
    random.seed(seed)
    stdev = math.sqrt(cstboltz * temperature / mass)
    v = []
    for _ in range(n):
        r1, r2 = random.random(), random.random()
        v.append([math.sqrt(-2*math.log(r1))*math.cos(r2)*stdev,
                  math.sqrt(-2*math.log(r1))*math.sin(0.5*r2)*stdev])
    vxt = sum(a[0] for a in v)/n; vyt = sum(a[1] for a in v)/n
    for a in v: a[0] -= vxt; a[1] -= vyt
    _, tt = calc_temp(v, n, cstboltz, mass); sc = math.sqrt(temperature / tt)
    return [[a[0]*sc, a[1]*sc] for a in v]

def run_md(use_pbc):
    coord = InitConf(nAtoms, box, Radius, qat, frac_neg, Seed)
    init = [list(c) for c in coord]
    vel = InitVel(nAtoms, Temperature, cstboltz, Mass)
    old = None; Et = []; wraps = 0
    for it in range(nrun):
        Ene, _, _ = Calc_Ene(coord, Epsilon, Rmin, dielec, cutoffsq, box, use_pbc)
        F = Calc_Force(coord, Epsilon, Rmin, dielec, cutoffsq, box, use_pbc)
        if it == 0:
            old = [list(c) for c in coord]; coord = Step1(coord, vel, F, dt, Mass)
        else:
            tmp = coord; coord = Verlet(coord, F, dt, old, Mass)
            vel = CalcVel(old, coord, dt); old = tmp
        Kin, _ = calc_temp(vel, nAtoms, cstboltz, Mass); Et.append(Ene + Kin)
        for pp in range(len(coord)):
            for i in range(2):
                if coord[pp][i] < 0:      coord[pp][i] += box[i]; old[pp][i] += box[i]; wraps += 1
                if coord[pp][i] > box[i]: coord[pp][i] -= box[i]; old[pp][i] -= box[i]; wraps += 1
    return Et, wraps, init, [list(c) for c in coord]

Et_on,  w_on,  init_on,  final_on  = run_md(True)
Et_off, w_off, init_off, final_off = run_md(False)
t = [i * dt for i in range(nrun)]
plt.figure(figsize=(9, 4.2))
plt.plot(t, Et_on,  color="#1f77b4", lw=1.5, label=f"with PBC (edge-crossings: {w_on})")
plt.plot(t, Et_off, color="#d62728", lw=1.0, ls="--", label=f"no PBC (edge-crossings: {w_off})")
plt.xlabel("time"); plt.ylabel("total energy")
plt.title("20-particle MD: PBC on vs off are indistinguishable (droplet never touches the walls)")
plt.legend(); plt.tight_layout(); plt.show()
print(f"with PBC: Etot {Et_on[0]:.1f} -> {Et_on[-1]:.1f}   ({100*abs(Et_on[-1]-Et_on[0])/abs(Et_on[0]):.1f}% drift)")
print(f"no  PBC: Etot {Et_off[0]:.1f} -> {Et_off[-1]:.1f}   ({100*abs(Et_off[-1]-Et_off[0])/abs(Et_off[0]):.1f}% drift)")

Here is what the system actually looks like — discs to scale (`Radius=25`),
white $=$ positive charge, dark $=$ negative. The particles are spread across the box
(not a tidy central droplet), but over these 2000 steps they barely move: the initial
and final configurations are almost the same, and the **PBC and no-PBC final
configurations are visually identical**. Crucially, **no particle touches a wall** —
nothing wraps — which is why the two runs never diverge.

In [ ]:
from matplotlib.patches import Circle, Rectangle
fig, ax = plt.subplots(1, 3, figsize=(12.5, 4.6))
panels = ((ax[0], init_on,  "initial"),
          (ax[1], final_on, f"final — with PBC"),
          (ax[2], final_off, f"final — no PBC"))
for a, coords, title in panels:
    a.add_patch(Rectangle((0, 0), box[0], box[1], fc="#ccddff", ec="k", lw=1.2))
    for c in coords:
        a.add_patch(Circle((c[0], c[1]), Radius, fc=charge_color(c[2]), ec="k", lw=0.5))
    a.set_xlim(-10, box[0]+10); a.set_ylim(-10, box[1]+10); a.set_aspect("equal")
    a.set_xticks([]); a.set_yticks([]); a.set_title(title, fontsize=10)
fig.suptitle(f"20-particle system after {nrun} steps — PBC and no-PBC coincide "
             "(white +, dark −)", fontsize=12)
plt.tight_layout(); plt.show()

def wall_gap(coords):
    return min(min(c[0], box[0]-c[0], c[1], box[1]-c[1]) for c in coords)
disp = max(math.hypot(a[0]-b[0], a[1]-b[1]) for a, b in zip(init_on, final_on))
print(f"closest any particle gets to a wall:  final {wall_gap(final_on):.0f}  "
      f"(> 0, so nothing wraps)")
print(f"largest net displacement of any particle over the run: {disp:.0f}  (caged, vibrating in place)")
print(f"PBC vs no-PBC final positions differ by at most "
      f"{max(math.hypot(a[0]-b[0], a[1]-b[1]) for a, b in zip(final_on, final_off)):.2f}")

## 7. Take-home messages

- **PBC = tile the box infinitely + interact with the nearest image** (minimum-image
  convention). It lets a small box mimic a large, homogeneous system.
- **Dropping the minimum-image convention only changes physics at the boundary.**
  A pair that would interact *across* a wall is either missed entirely or feels a
  discontinuous jump when a particle wraps (Demos A and B).
- **If nothing crosses (or strongly interacts across) a boundary, PBC is
  irrelevant.** In the standard 20-particle run the particles are caged and vibrate
  in place without ever reaching a wall, so PBC on/off are identical (Demo C) — even
  though the minimum-image convention is technically active. To *see* the effect you
  need particles that actually cross or straddle the boundary: a dense fluid, a gas
  that fills the box, a smaller box, or particles given a net drift toward a wall.
- **Watch out for confounds.** The original no-PBC script's big apparent difference
  was a kinetic-energy bug, not the missing PBC — always isolate the one variable you
  mean to study (here: run the *same* code with a single `use_pbc` switch).